In [6]:
import spatial_sdm as sdm

In [2]:
# ============================================================
# Spatial Durbin Model (SDM) – Main Execution Script
# ============================================================

# spatial_sdm.py
"""
Spatial SDM (Spatial Durbin Model) for panel data in PyMC.
...
"""
from __future__ import annotations

import numpy as np
import pandas as pd
import pymc as pm
import pytensor.tensor as pt

# ------------------------------------------------------------
# 1) Load data
# ------------------------------------------------------------
df_sorted = pd.read_excel("/content/df_sorted.xlsx")
W_cul04_raw = pd.read_excel("/content/W_cul04.xlsx")
W_cul06_raw = pd.read_excel("/content/W_cul06.xlsx")
W_cul05_raw = pd.read_excel("/content/W_cul05.xlsx")
W_geo_raw = pd.read_excel("/content/W_geo.xlsx")
W_trade_raw = pd.read_excel("/content/W_trade.xlsx")



# ------------------------------------------------------------
# 2) Clean spatial weights and align with panel data
# ------------------------------------------------------------
W_cul04 = sdm.prepare_W_from_excel(W_cul04_raw)
W_cul06 = sdm.prepare_W_from_excel(W_cul06_raw)
W_cul05 = sdm.prepare_W_from_excel(W_cul05_raw)
W_geo = sdm.prepare_W_from_excel(W_geo_raw)
W_trade = sdm.prepare_W_from_excel(W_trade_raw)


df, W = sdm.align_df_and_W(df_sorted, W_cul04)

# Ensure correct sorting (VERY IMPORTANT)
df = df.sort_values(["year", "country"]).copy()
df["year"] = df["year"].astype(int)

# ------------------------------------------------------------
# 3) Sanity checks
# ------------------------------------------------------------
years = sorted(df["year"].unique())
countries = sorted(df["country"].unique())

print("Number of countries (N):", len(countries))
print("Number of years (T):", len(years))
print("Number of observations (NT):", df.shape[0])
print("Expected NT = N * T:", len(countries) * len(years))
print("Balanced panel:", df.shape[0] == len(countries) * len(years))
print("W shape:", W.shape)

# ------------------------------------------------------------
# 4) Fit SDM on the full dataset
# ------------------------------------------------------------



trace_full, countries_sorted, years_sorted, W_base = sdm.run_sdm_model(
    df, W,
    draws=2000,
    tune=2000,
    chains=4,
    target_accept=0.95,
    random_seed=123,
    cores=None,
    progressbar=True,
)

print("SDM estimation completed.")



 Progress                    Draws   Divergences   Step size   Grad evals   Sampling Speed   Elapsed   Remaining  
 ───────────────────────────────────────────────────────────────────────────────────────────────────────────────── 
  ━━━━━━━━━━━━━━━━━━━━━━━━━   4000    0             0.004       1023         3.37 draws/s     0:19:47   0:00:00    
  ━━━━━━━━━━━━━━━━━━━━━━━━━   4000    0             0.003       2047         1.78 draws/s     0:37:28   0:00:00    
  ━━━━━━━━━━━━━━━━━━━━━━━━━   4000    0             0.005       2047         1.21 draws/s     0:54:55   0:00:00    
  ━━━━━━━━━━━━━━━━━━━━━━━━━   4000    0             0.004       1023         1.10 s/draws     1:13:04   0:00:00

SDM estimation completed.


In [3]:
import importlib
import spatial_sdm as sdm
importlib.reload(sdm)

summary_main, df_alpha, df_time_alpha = sdm.summarize_sdm_trace(
    trace_full, countries_sorted, years_sorted, round_to=4, verbose=True
)

summary_main
df_alpha.head()
df_time_alpha.head()


Main coefficients summary:
             mean      sd  hdi_3%  hdi_97%  mcse_mean  mcse_sd   ess_bulk  \
rho      -0.7984  0.1137 -0.9489  -0.5935     0.0012   0.0014  7759.6558   
beta[0]   0.2140  0.1583 -0.1012   0.4946     0.0018   0.0017  7755.3127   
beta[1]   0.5689  0.1163  0.3477   0.7848     0.0016   0.0012  5536.6291   
beta[2]   0.3038  0.1093  0.1012   0.5104     0.0015   0.0011  5369.8049   
beta[3]   0.3897  0.1785  0.0656   0.7484     0.0025   0.0020  5080.9681   
gamma[0]  2.2647  0.3129  1.6775   2.8460     0.0040   0.0033  5960.2352   
gamma[1]  0.3508  0.5084 -0.5875   1.3188     0.0068   0.0054  5598.6181   
gamma[2]  0.9515  0.4378  0.1336   1.7609     0.0058   0.0054  5624.1433   
gamma[3] -0.2271  0.6595 -1.4708   1.0198     0.0103   0.0076  4114.2063   
sigma     0.3058  0.0178  0.2744   0.3402     0.0002   0.0002  6053.1641   

           ess_tail   r_hat  
rho       5308.1953  1.0007  
beta[0]   6124.8908  1.0011  
beta[1]   5655.5224  1.0000  
beta[2]   5027.

,year,time_alpha_mean,time_alpha_sd
0,2002,-0.578690,0.484331
1,2003,-0.469974,0.470184
2,2004,-0.114804,0.474972
3,2005,-0.256979,0.474840
4,2006,0.051986,0.488531


In [4]:
import numpy as np
import pandas as pd
import arviz as az
import spatial_sdm as sdm

# ------------------------------------------------------------
# Helper: build X, WX, indices (must match model ordering)
# ------------------------------------------------------------
def build_design_mats(df_sorted, W_base):
    """
    Build X, WX, y, country_idx, year_idx consistent with run_sdm_model ordering.

    Assumes df_sorted is sorted by ['year','country'] and panel is balanced.
    """
    df_sorted = df_sorted.sort_values(["year", "country"]).copy()
    df_sorted["year"] = df_sorted["year"].astype(int)

    years_sorted = sorted(df_sorted["year"].unique())
    countries_sorted = sorted(df_sorted["country"].unique())
    T, N = len(years_sorted), len(countries_sorted)

    country_to_idx = {c: i for i, c in enumerate(countries_sorted)}
    year_to_idx = {y: i for i, y in enumerate(years_sorted)}

    country_idx = df_sorted["country"].map(country_to_idx).to_numpy("int32")
    year_idx = df_sorted["year"].map(year_to_idx).to_numpy("int32")

    X = df_sorted[['GDP','Political Stability','exchange rate',"Rule of Law: Estimate"]].to_numpy("float64")
    y = df_sorted["inbound"].to_numpy("float64")

    NT, K = X.shape
    assert NT == N * T, "Balanced panel required (NT == N*T)."

    WX = np.zeros((NT, K), dtype="float64")
    for t in range(T):
        X_t = X[t*N:(t+1)*N, :]
        WX[t*N:(t+1)*N, :] = W_base @ X_t

    return X, WX, y, country_idx, year_idx, countries_sorted, years_sorted, N, T


# ------------------------------------------------------------
# 1) Build matrices for prediction diagnostics
# ------------------------------------------------------------
X, WX, y, country_idx, year_idx, countries_sorted, years_sorted, N, T = build_design_mats(df, W_base)

# ------------------------------------------------------------
# 2) Posterior mean prediction and MSE by country
# ------------------------------------------------------------
# If you still have your compute_sdm_predictions() function in the notebook, use it.
# Otherwise, here is a compact version:

def posterior_mean_predictions(trace, X, WX, country_idx, year_idx, W_base):
    """
    Compute posterior mean of E[y | params] for each observation:
        y_hat_t = (I - rho W)^(-1) * XB_t
    """
    post = trace.posterior
    beta = post["beta"].mean(dim=("chain","draw")).values
    gamma = post["gamma"].mean(dim=("chain","draw")).values
    alpha = post["alpha"].mean(dim=("chain","draw")).values
    time_alpha = post["time_alpha"].mean(dim=("chain","draw")).values
    rho = float(post["rho"].mean(dim=("chain","draw")).values)

    XB = X @ beta + WX @ gamma + alpha[country_idx] + time_alpha[year_idx]

    S = np.linalg.inv(np.eye(W_base.shape[0]) - rho * W_base)

    y_hat = np.zeros_like(XB)
    for t in range(T):
        y_hat[t*N:(t+1)*N] = S @ XB[t*N:(t+1)*N]

    return y_hat

y_hat = posterior_mean_predictions(trace_full, X, WX, country_idx, year_idx, W_base)

mse_by_country = {}
for c_i, c in enumerate(countries_sorted):
    mask = (country_idx == c_i)
    mse_by_country[c] = float(np.mean((y[mask] - y_hat[mask])**2))

mse_df = pd.DataFrame({"country": list(mse_by_country.keys()),
                       "mse": list(mse_by_country.values())}).sort_values("mse")

print("MSE by country (lower is better):")
display(mse_df)

# Optional: save
mse_df.to_csv("mse_by_country.csv", index=False)


# ------------------------------------------------------------
# 3) Moran's I on SDM residuals by year
# ------------------------------------------------------------
# We need a PySAL weights object + residuals per year
from libpysal.weights import W as W_pysal
from esda.moran import Moran

# Build PySAL W object from W_base
neighbors = {i: list(np.where(W_base[i] > 0)[0]) for i in range(N)}
weights = {i: W_base[i, neighbors[i]] for i in range(N)}
W_obj = W_pysal(neighbors, weights)

# Extract posterior means
post = trace_full.posterior
rho_mean = float(post["rho"].mean(dim=("chain","draw")).values)
beta_mean = post["beta"].mean(dim=("chain","draw")).values
gamma_mean = post["gamma"].mean(dim=("chain","draw")).values
alpha_mean = post["alpha"].mean(dim=("chain","draw")).values
time_alpha_mean = post["time_alpha"].mean(dim=("chain","draw")).values

# Linear component XB
XB = (X @ beta_mean) + (WX @ gamma_mean) + alpha_mean[country_idx] + time_alpha_mean[year_idx]

# SDM "structural" residuals per year:
# y - rho*W y - XB
resid = np.zeros_like(y)
for t in range(T):
    y_t = y[t*N:(t+1)*N]
    XB_t = XB[t*N:(t+1)*N]
    resid[t*N:(t+1)*N] = y_t - rho_mean * (W_base @ y_t) - XB_t

# Moran's I by year
morans_rows = []
for t in range(T):
    r_t = resid[t*N:(t+1)*N]
    mi = Moran(r_t, W_obj)
    morans_rows.append({
        "year": int(years_sorted[t]),
        "moran_I": float(mi.I),
        "p_norm": float(mi.p_norm),      # normal approximation p-value
        "p_sim": float(mi.p_sim),        # permutation p-value (default perms in esda)
    })

morans_df = pd.DataFrame(morans_rows)
print("Moran's I of residuals by year:")
display(morans_df)

# Optional: save
morans_df.to_csv("moransI_residuals_by_year.csv", index=False)


MSE by country (lower is better):


,country,mse
10,Türkiye,0.028992
2,Egypt,0.037969
4,India,0.046836
5,Iran,0.048827
9,Saudi Arabia,0.063281
6,Italy,0.070791
0,Azerbaijan,0.074591
1,China,0.091931
7,Kazakhstan,0.116112
3,Georgia,0.234815


Moran's I of residuals by year:


,year,moran_I,p_norm,p_sim
0,2002,-0.213300,0.239616,0.058
1,2003,-0.166201,0.492018,0.250
2,2004,-0.156500,0.557596,0.278
3,2005,-0.204778,0.276817,0.134
4,2006,-0.162775,0.514696,0.326
5,2007,-0.012950,0.366263,0.201
6,2008,-0.029247,0.462736,0.212
7,2009,-0.190663,0.346706,0.055
8,2010,-0.144795,0.641982,0.277
9,2011,-0.144963,0.640732,0.304


In [ ]:
# ------------------------------------------------------------
# 5) (Optional) Save posterior samples for reproducibility
# ------------------------------------------------------------
import arviz as az
az.to_netcdf(trace_full, "trace_sdm_full.nc")

# ------------------------------------------------------------
# 6) Leave-One-Country-Out (LOCO) cross-validation
# ------------------------------------------------------------
# NOTE: This is computationally expensive.
# Start with small draws/tune for testing.

res_loco = sdm.loco_cv(
    df=df,
    W_df=W,
    draws=1000,           # increase after testing
    tune=1000,
    chains=2,
    target_accept=0.95,
    random_seed=123,
    cores=None,
    progressbar=False,
)

print(res_loco["status"].value_counts())
print(res_loco.head())

# Save LOCO results
res_loco.to_csv("loco_results.csv", index=False)

print("LOCO cross-validation completed.")
